# LLM

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

1. OpenAI
2. Ollama - not practicing it since mac os - ventura - 13.4 dont support it.
3. Hugging face 
4. Groq

In [ ]:
# OpenAI
from openai import OpenAI
print(len([m.id for m in OpenAI().models.list()]))

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-5.4-mini",temperature=0)
answer = llm.invoke("What is current date?")
print(answer)

In [ ]:
from huggingface_hub import HfApi
from langchain_huggingface import HuggingFaceEndpoint , ChatHuggingFace

api = HfApi()
models = api.list_models(pipeline_tag="text-generation",inference_provider='all',sort="downloads",limit=30)

print([m.id for m in models])

llm = HuggingFaceEndpoint(repo_id="Qwen/Qwen3-0.6B",temperature=0)
chat_model = ChatHuggingFace(llm=llm)
print(chat_model.invoke("How is Bangalore?"))

In [ ]:
from groq import Groq
from langchain_groq import ChatGroq

response = Groq().models.list()

print([m.id for m in response.data])

llm = ChatGroq(model="openai/gpt-oss-20b")
print(llm.invoke("Famous tourist place in Karnataka"))

In [ ]:
from langchain.chat_models import init_chat_model

llm_gr = init_chat_model(model="groq:openai/gpt-oss-20b")
print(llm_gr.invoke("Famous tourist place in Karnataka"))

In [ ]:
from langchain.chat_models import init_chat_model
llm_1 = init_chat_model(configurable_fields=("model","model_provider"))
print(llm_1.invoke("How is weather in Banglore",config={"configurable":{"model":"groq:openai/gpt-oss-20b"}}))

In [ ]:
for chunk in llm.stream("Write about AI in 200 words"):
    print(chunk.text, end = "|",flush=True)

# Prompt


In [ ]:
from langchain_core.prompts import PromptTemplate

promptTemp = PromptTemplate(input_variables=["name","age","message"],partial_variables={"appr_age":"32"},template="My name is {name},age is {age}.I have message :{message},{appr_age}")
print(promptTemp.invoke({"name":"vsh","age":"","message":"Be kind"}))

promptTemp = PromptTemplate.from_template(template="My name is {name},age is {age}.I have message :{message},{appr_age}")
promptTemp = promptTemp.partial(appr_age=32)
print(promptTemp.invoke({"name":"vsh","age":"","message":"Be kind"}))

chain = promptTemp | llm_gr
print(chain.invoke({"name":"vsh","age":18,"message":"Be kind"}))

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage,AIMessage,SystemMessage

messages =[
    ("system","You are designed for text gentation,your name is {assistantName}"),
    ("human","Hi, how are you?"),
    ("ai","Doing good! How can I help you today?")
]

chatTemplate = ChatPromptTemplate.from_messages(messages=messages)
print(chatTemplate.invoke({"assistantName":"aiAssistant"}))

messages =[
    SystemMessage(content="You are designed for text gentation,your name is {assistantName}"), # it prints as {assistantName} in print, because, it can process direct message
    HumanMessage(content = "Hi, how are you?"),
    AIMessage(content ="Doing good! How can I help you today?"),
    ("human","{query}")
]
chatTemplate = ChatPromptTemplate.from_messages(messages=messages)
# print(chatTemplate.invoke({"assistantName":"aiAssistant"}))

chain = chatTemplate | llm_gr
print(chain.invoke({"assistantName":"aiAssistant","query":"steps to learn ai"}).content)


In [ ]:
from langchain_core.prompts import ChatPromptTemplate
# Import the TEMPLATE versions, not the raw message versions
from langchain_core.prompts import SystemMessagePromptTemplate, HumanMessagePromptTemplate

chatTemplate = ChatPromptTemplate.from_messages([
    SystemMessagePromptTemplate.from_template("Your name is {assistantName}"),
    HumanMessagePromptTemplate.from_template("Hello, my name is {userName}")
])

print(chatTemplate.invoke({"assistantName": "Jarvis", "userName": "Tony"}))


In [ ]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder

history = [
    ("human","How is Bangalore?"),
    ("ai","It is good place.")
]
messages =[
    ("system","You are designed for text gentation,your name is {assistantName}"),
    ("human","Hi, how are you?"),
    ("ai","Doing good! How can I help you today?"),
    MessagesPlaceholder(variable_name="history")
]

chatTemplate = ChatPromptTemplate.from_messages(messages=messages)
print(chatTemplate.invoke({"assistantName":"aiAssistant","history":history}))

In [ ]:
from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate

examples = [
    {"input":"small","output":"big"},
    {"input":"sharp","output":"dull"},
    {"input":"on","output":"off"}

]

base_template = PromptTemplate(input_variables=["input","output"],template = "input:{input}\noutput:")
fewshot_template = FewShotPromptTemplate(examples=examples,example_prompt=base_template,prefix="Give the antonyms for given word",
                                         suffix = "input:{adjective}\noutput:",input_variables = ["adjective"])
print(fewshot_template.invoke({"adjective":"dark"}).text)

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate,ChatPromptTemplate

examples = [
    {"input":"small","output":"big"},
    {"input":"sharp","output":"dull"},
    {"input":"on","output":"off"}
]

base_template = ChatPromptTemplate.from_messages([
    ("human","{input}"),
    ("ai","{output}")
])

fewshotchat_tempalte = FewShotChatMessagePromptTemplate(examples=examples,example_prompt=base_template)

final_tempalte = ChatPromptTemplate.from_messages([
    ("system","You are designed for giving opposite words"),
    fewshotchat_tempalte,
    ("human","{adjective}")
])

print(final_tempalte.invoke({"adjective":"dark"}).to_messages())

print([message.content for message in final_tempalte.invoke({"adjective":"dark"}).to_messages()])

In [ ]:
chain = final_tempalte | chat_model

answer = chain.invoke({"adjective":"dark"})


In [ ]:
answer.content

In [ ]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chain = final_tempalte | chat_model | StrOutputParser()   # returns a plain string instead of AIMessage

full_chain = RunnablePassthrough.assign(output=chain)

result = full_chain.invoke({"adjective": "dark"})
print(result)
# {'adjective': 'dark', 'output': 'light'}

print(f"Input: {result['adjective']}  ->  Output: {result['output']}")

In [ ]:
from langchain_core.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate

examples = [
    {"input":"small","output":"big"},
    {"input":"sharp","output":"dull"},
    {"input":"on","output":"off"}
]

base_template = ChatPromptTemplate.from_messages(messages=[
    ("human","{input}"),
    ("ai","{output}")
])

fewShot_msg = FewShotChatMessagePromptTemplate(examples=examples,example_prompt=base_template)

final_tempalte = ChatPromptTemplate.from_messages(messages=[
    ("system","Its is opposite word generator"),
    fewShot_msg,
    ("human","{adjective}")
])

chain = final_tempalte | llm_1

print(chain.invoke({"adjective":"dark"},config={"configurable":{"model":"groq:openai/gpt-oss-20b"}}).content)

In [ ]:
#dynamic example selection

from langchain_core.prompts import PromptTemplate,FewShotPromptTemplate
from langchain_classic.vectorstores import FAISS
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_core.example_selectors import SemanticSimilarityExampleSelector

examples = [
    {"input":"small","output":"big"},
    {"input":"sharp","output":"dull"},
    {"input":"on","output":"off"}
]

embeddings = HuggingFaceEmbeddings(model_name = "sentence-transformers/all-MiniLM-L6-v2")
example_Selector = SemanticSimilarityExampleSelector.from_examples(examples=examples, embeddings=embeddings,vectorstore_cls=FAISS,k=2)
base_template = PromptTemplate.from_template("input:{input}\nouput:{output}")
fewShot_template = FewShotPromptTemplate(example_selector=example_Selector,example_prompt=base_template,prefix="Give the antonyms for given word",suffix="input:{adjective}\noutput:",input_variables=["adjective"])

print(fewShot_template.invoke({"adjective":"bright"}).text)

chain = fewShot_template | llm_gr

print(chain.invoke({"adjective":"bright"}))


In [ ]:
from langchain_core.prompts import ChatPromptTemplate,FewShotChatMessagePromptTemplate

examples = [
    {"input":"small","output":"big"},
    {"input":"sharp","output":"dull"},
    {"input":"on","output":"off"}
]

chat_template = ChatPromptTemplate.from_messages([
    ("human","{input}"),
    ("ai","{output}")
])

fewShot_template = FewShotChatMessagePromptTemplate(examples=examples,example_prompt=chat_template)
final_template = ChatPromptTemplate.from_messages(messages=[
    ("system","You are designed for generating opposite words"),
    fewShot_template,
    ("human","{adjective}")
])
print(final_template.invoke({"adjective":"silent"}).to_messages())

chain = final_template | llm_gr

print(chain.invoke({"adjective":"silent"}))



# Output parsers

with_structured_output <br>
get_format_instructions() <br>
parse() and invoke() <br>

1. StrOutputParser
2. PydanticOutputParser
3. CommaSeparatedListOutputParser
4. JsonOutputParser

In [25]:
from langchain_core.output_parsers import StrOutputParser

chain = final_template | llm_gr | StrOutputParser()  # returns <class 'langchain_core.messages.base.TextAccessor'>
print(chain.invoke({"adjective":"silent"}))
print(type(chain.invoke({"adjective":"silent"})))

NameError: name 'final_template' is not defined

In [ ]:
parser = StrOutputParser()
final_template.partial({"parser":parser.get_format_instructions()})

In [ ]:
from pydantic import BaseModel, Field
from pydantic_settings import BaseSettings
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

messages =[
    ("system","You are designed for giving cricketer info"),
    ("human","Hi, how are you?"),
    ("ai","Doing good! How can I help you today?"),
     ("human","Please tell me about cricketer {cricketerName}"),
]

chatTemplate = ChatPromptTemplate.from_messages(messages=messages)
# print(chatTemplate.invoke({"assistantName":"aiAssistant"}))

class Cricketer(BaseModel):
    name: str = Field(description="Full name")
    country: str = Field(description="Country represented")
    role: str = Field(description="Batsman, bowler, all-rounder, etc.")
    ipl_team: str = Field(description="Current or most recent IPL team")

parser = PydanticOutputParser(pydantic_object=Cricketer)

chatTemplate = ChatPromptTemplate.from_messages(messages).partial(
    format_instructions=parser.get_format_instructions()
)

chain = chatTemplate | llm_gr | parser

print(chain.invoke({"cricketerName":"Virat Kohli"}))





In [ ]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser

class Cricketer(BaseModel):
    name: str = Field(description="Full name")
    country: str = Field(description="Country represented")
    role: str = Field(description="Batsman, bowler, all-rounder, etc.")
    ipl_team: str = Field(description="Current or most recent IPL team")

parser = PydanticOutputParser(pydantic_object=Cricketer)

chatTemplate = ChatPromptTemplate.from_messages([
    ("system", "Return ONLY valid JSON. No markdown, no extra text.\n{format_instructions}"),
    ("human", "Give details about the cricketer {cricketerName}."),
]).partial(format_instructions=parser.get_format_instructions())

chain = chatTemplate | llm_gr | parser
print(chain.invoke({"cricketerName": "Virat Kohli"}))

In [23]:
structured_llm = llm_gr.with_structured_output(Cricketer)
chatTemplate = ChatPromptTemplate.from_messages([
    ("system", "Respond only in JSON with keys: name, country, role, ipl_team."),
    ("human", "Give details about the cricketer {cricketerName}."),
])

chain = chatTemplate | structured_llm   # format_instructions no longer needed
print(chain.invoke({"cricketerName": "Virat Kohli"}))

name='Virat Kohli' country='India' role='Batsman' ipl_team='Royal Challengers Bangalore'


In [ ]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

messages = [
    ("system", "You are designed for giving cricketer info. Respond only with JSON, no prose or markdown."),
    ("human", "Hi, how are you?"),
    ("ai", "Doing good! How can I help you today?"),
    ("human", "Please tell me about cricketer {cricketerName}.\n\n{format_instructions}"),
]

class Cricketer(BaseModel):
    name: str = Field(description="Full name of the cricketer")
    age: int = Field(description="Current age in years")
    highestScore: str = Field(description="Highest score in international cricket, e.g. '254*'")

parser = PydanticOutputParser(pydantic_object=Cricketer)

chatTemplate = ChatPromptTemplate.from_messages(messages).partial(
    format_instructions=parser.get_format_instructions()
)

chain = chatTemplate | llm_gr | parser
result = chain.invoke({"cricketerName": "Virat Kohli"})
print(result.model_dump_json())

In [ ]:
result = parser.parse('{"name": "Virat Kohli", "age": 37, "highestScore": "254*"}')
print(type(result))

In [ ]:
from pydantic import BaseModel , Field
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import CommaSeparatedListOutputParser

parser = CommaSeparatedListOutputParser()
messages = [
    ("system", "You are designed for giving cricketer info. Respond only with JSON, no prose or markdown."),
    ("human", "Hi, how are you?"),
    ("ai", "Doing good! How can I help you today?"),
    ("human", "Plese give list of 10 {country} cricketers.\n\n{foramt_instructions}"),
]
chatTemplate = ChatPromptTemplate.from_messages(messages).partial(foramt_instructions = parser.get_format_instructions())

chain = chatTemplate | llm_gr | parser

print(chain.invoke({"country":"Indian"}))


In [24]:
from pydantic import BaseModel, Field
from langchain_core.output_parsers import JsonOutputParser


class Cricketer(BaseModel):
    name: str = Field(description="Full name of the cricketer")
    age: int = Field(description="Current age in years")
    highestScore: str = Field(description="Highest score in international cricket, e.g. '254*'")

class CricketersList(BaseModel):
    playersList : list[Cricketer]

parser = JsonOutputParser(pydantic_object=CricketersList)
messages = [
    ("system", "You are designed for giving cricketer info. Respond only with JSON, no prose or markdown."),
    ("human", "Hi, how are you?"),
    ("ai", "Doing good! How can I help you today?"),
    ("human", "Plese give list of 10 {country} cricketers.\n\n{foramt_instructions}"),
]
chatTemplate = ChatPromptTemplate.from_messages(messages).partial(foramt_instructions = parser.get_format_instructions())

chain = chatTemplate | llm_gr | parser

# answer = chain.invoke({"country":"Indian"})

for chunk in chain.stream({"country":"Indian"}):
    print(chunk)







{}
{'playersList': []}
{'playersList': [{}]}
{'playersList': [{'name': ''}]}
{'playersList': [{'name': 'Vir'}]}
{'playersList': [{'name': 'Virat'}]}
{'playersList': [{'name': 'Virat Koh'}]}
{'playersList': [{'name': 'Virat Kohli'}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': ''}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {'name': ''}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {'name': 'R'}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {'name': 'Roh'}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {'name': 'Rohit'}]}
{'playersList': [{'name': 'Virat Kohli', 'age': 37, 'highestScore': '183'}, {'name': 'Rohit Sharma'}]}
{'pla